# Demand forecast — temporal validation

Predicting daily demand from lagged history. Validation here is temporal: metrics come from forecasting later days using only information from earlier days, never the reverse.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
rng = np.random.default_rng(13)
dates = pd.date_range('2024-01-01', periods=730, freq='D')
n = len(dates)
trend = np.linspace(100, 160, n)
weekly = 15 * np.sin(2 * np.pi * np.arange(n) / 7)
noise = rng.normal(0, 5, n)
ar = np.zeros(n)
for t in range(1, n):
    ar[t] = 0.6 * ar[t - 1] + noise[t]
demand = trend + weekly + ar
df = pd.DataFrame({'date': dates, 'demand': demand.round(1)})

In [3]:
df['demand_lag1'] = df['demand'].shift(1)
df['demand_lag7'] = df['demand'].shift(7)
df['demand_ma14'] = df['demand'].rolling(14).mean()
df['dow'] = df['date'].dt.dayofweek
df = df.dropna().reset_index(drop=True)

In [4]:
feature_cols = ['demand_lag1', 'demand_lag7', 'demand_ma14', 'dow']
X = df[feature_cols]
y = df['demand']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=40)
model = Ridge(alpha=1.0)
model.fit(X_tr, y_tr)
pred = model.predict(X_te)
mae = mean_absolute_error(y_te, pred)
r2 = r2_score(y_te, pred)
print(f"mae={mae:.3f}  r2={r2:.3f}")

mae=4.820  r2=0.917


In [5]:
cv = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(Ridge(alpha=1.0), X, y, cv=cv, scoring='neg_mean_absolute_error')
print(f"cv neg_mae: {scores.mean():.3f} +/- {scores.std():.3f}")

cv neg_mae: -5.369 +/- 0.383


The model reaches the MAE and R^2 printed above on the held-out split, with cross-validated MAE shown alongside.